# 千问 AI 平台模型调用演示

在 VS Code 中打开项目文件夹，选用项目的 Python 内核，然后按顺序运行三个代码块。配置从项目目录的 `.env` 读取。

## 1. 读取配置

只显示模型名和接口地址。

In [ ]:
from pathlib import Path

env_file = Path(".env")
if not env_file.is_file():
    raise FileNotFoundError("请在 VS Code 中打开项目文件夹，再运行此 Notebook")

required = ("DASHSCOPE_API_KEY", "DASHSCOPE_BASE_URL", "DASHSCOPE_MODEL")
config = {}
for line in env_file.read_text(encoding="utf-8-sig").splitlines():
    name, separator, value = line.partition("=")
    if separator and name.strip() in required:
        config[name.strip()] = value.strip().strip("\"'")

missing = [name for name in required if not config.get(name)]
if missing:
    raise ValueError(f".env 缺少配置：{', '.join(missing)}")

model = config["DASHSCOPE_MODEL"]
base_url = config["DASHSCOPE_BASE_URL"].rstrip("/")
print(f"模型：{model}")
print(f"接口：{base_url}")

## 2. 构造请求

可以修改 `prompt`，再运行这一块和下一块。

In [ ]:
import json
from urllib.request import Request, urlopen
prompt = "请用一句话介绍你自己"
payload = {
    "model": model,
    "messages": [{"role": "user", "content": prompt}],
    "enable_thinking": False,
    "max_tokens": 64,
}
request = Request(
    f"{base_url}/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={
        "Authorization": f"Bearer {config['DASHSCOPE_API_KEY']}",
        "Content-Type": "application/json",
    },
    method="POST",
)

## 3. 调用模型并查看回复

每次运行本单元格都会发起一次 API 调用。

In [ ]:
with urlopen(request, timeout=30) as response:
    result = json.load(response)

reply = result["choices"][0]["message"]["content"]
if not reply or not reply.strip():
    raise ValueError("模型没有返回正文")

print(f"实际返回模型：{result.get('model', model)}")
print(f"回复：{reply.strip()}")